# Week 3 Practical Lab: Building a Simple Content-Based Recommender
### You will build the following workflow:
Movie Features → User Preferences → Recommendation Scores → Top Recommendations

**1. Import pandas.**

In [62]:
import pandas as pd

**2. Create the Movie Dataset.**

In [63]:
movies = pd.DataFrame({
"movie_id": [1,2,3,4,5,6,7,8,9,10],
"title": [
"Galaxy Mission","Love in Nairobi","Final Battle","Family Weekend",
"Future World","Broken Hearts","City Detectives","Space Warriors",
"Wedding Weekend","Hidden Truth"
],
"Action": [1,0,1,0,1,0,1,1,0,0],
"Comedy": [0,0,0,1,0,0,0,0,1,0],
"Drama": [0,1,0,0,0,1,1,0,0,1],
"Romance": [0,1,0,0,0,1,0,0,1,0],
"SciFi": [1,0,0,0,1,0,0,1,0,0]
})
movies

,movie_id,title,Action,Comedy,Drama,Romance,SciFi
0,1,Galaxy Mission,1,0,0,0,1
1,2,Love in Nairobi,0,0,1,1,0
2,3,Final Battle,1,0,0,0,0
3,4,Family Weekend,0,1,0,0,0
4,5,Future World,1,0,0,0,1
5,6,Broken Hearts,0,0,1,1,0
6,7,City Detectives,1,0,1,0,0
7,8,Space Warriors,1,0,0,0,1
8,9,Wedding Weekend,0,1,0,1,0
9,10,Hidden Truth,0,0,1,0,0


In [64]:
movies.to_csv("movies.csv", index=False)

In [65]:
movies_check = pd.read_csv("movies.csv")

movies_check.head()

,movie_id,title,Action,Comedy,Drama,Romance,SciFi
0,1,Galaxy Mission,1,0,0,0,1
1,2,Love in Nairobi,0,0,1,1,0
2,3,Final Battle,1,0,0,0,0
3,4,Family Weekend,0,1,0,0,0
4,5,Future World,1,0,0,0,1


**3. Explore the Movie Dataset.**

In [66]:
movies.head()
movies.shape
movies.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10 entries, 0 to 9
Data columns (total 7 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   movie_id  10 non-null     int64 
 1   title     10 non-null     object
 2   Action    10 non-null     int64 
 3   Comedy    10 non-null     int64 
 4   Drama     10 non-null     int64 
 5   Romance   10 non-null     int64 
 6   SciFi     10 non-null     int64 
dtypes: int64(6), object(1)
memory usage: 692.0+ bytes


Answers:

Q1. The dataset contains 10 movies, with 10 rows and 7 columns.

Q2. It has 5 genre features: Action, Comedy, Drama, Romance, and SciFi. The `movie_id` and `title` columns identify each movie.

Q3. Each row represents one movie, containing its ID, title, and five binary genre values (0 or 1). These genre values describe the movie's characteristics and form its item profile.

**4. Separate the Movie Features.**

In [67]:
features = ["Action", "Comedy", "Drama", "Romance", "SciFi"]
movie_features = movies[features]
movie_features

,Action,Comedy,Drama,Romance,SciFi
0,1,0,0,0,1
1,0,0,1,1,0
2,1,0,0,0,0
3,0,1,0,0,0
4,1,0,0,0,1
5,0,0,1,1,0
6,1,0,1,0,0
7,1,0,0,0,1
8,0,1,0,1,0
9,0,0,1,0,0


**5. Inspect One Item Profile**

In [68]:
movies.loc[
movies["title"] == "Galaxy Mission"
]

,movie_id,title,Action,Comedy,Drama,Romance,SciFi
0,1,Galaxy Mission,1,0,0,0,1


Answer:
     
1 means the movie has that genre, and 0 means it does not. Galaxy Mission has Action = 1 and SciFi = 1, and 0 for Comedy, Drama and Romance, so it is an action sci-fi movie with an item profile of 1, 0, 0, 0, 1.

**6. Create User Ratings**

In [69]:
ratings = pd.DataFrame({
"movie_id": [1, 2, 3, 5],
"rating": [5, 2, 4, 5]
})
ratings

,movie_id,rating
0,1,5
1,2,2
2,3,4
3,5,5


**7.  Merge the Movie Features and User Ratings**

In [70]:
user_movies = ratings.merge(
movies,
on="movie_id"
)
user_movies

,movie_id,rating,title,Action,Comedy,Drama,Romance,SciFi
0,1,5,Galaxy Mission,1,0,0,0,1
1,2,2,Love in Nairobi,0,0,1,1,0
2,3,4,Final Battle,1,0,0,0,0
3,5,5,Future World,1,0,0,0,1


**8. Explore the User Profile**

In [71]:
user_movies[
["title", "rating", "Action", "Comedy", "Drama", "Romance", "SciFi"]
]

,title,rating,Action,Comedy,Drama,Romance,SciFi
0,Galaxy Mission,5,1,0,0,0,1
1,Love in Nairobi,2,0,0,1,1,0
2,Final Battle,4,1,0,0,0,0
3,Future World,5,1,0,0,0,1


#### Observation Before Calculating

The user seems to enjoy **Action and SciFi movies** the most. They rated *Galaxy Mission* and *Future World* 5 stars, and both belong to the Action and SciFi genres. They also rated *Final Battle*, an Action movie, 4 stars. In contrast, *Love in Nairobi*, which features Drama and Romance, received only 2 stars. This suggests that the user may prefer action-packed and science-fiction movies over romantic dramas.

**9. Select rated movies and their features**

In [72]:
rated_features = user_movies[features]
user_ratings = user_movies["rating"]
rated_features
user_ratings

0    5
1    2
2    4
3    5
Name: rating, dtype: int64

**10. Weight Features Using Ratings**

In [73]:
weighted_features = rated_features.mul(
user_ratings,
axis=0
)
weighted_features

,Action,Comedy,Drama,Romance,SciFi
0,5,0,0,0,5
1,0,0,2,2,0
2,4,0,0,0,0
3,5,0,0,0,5


#### Interpretation:

Movies with higher ratings (SciFi) contribute more strongly to the user's preference profile.

**11. Build the User Profile**

In [74]:
user_profile = weighted_features.sum()

user_profile

Action     14
Comedy      0
Drama       2
Romance     2
SciFi      10
dtype: int64

This adds the weighted genre scores across all the movies User A has rated.

**12. Interpret the User Profile**

In [75]:
user_profile.sort_values(
ascending=False
)

Action     14
SciFi      10
Drama       2
Romance     2
Comedy      0
dtype: int64

Answer:

The two genres I prefer most are Action, with a score of 14, and SciFi, with a score of 10. Action gets 14 from the ratings of 5, 4, and 5, while SciFi gets 10 from two ratings of 5 each. Drama and Romance each score 2, while Comedy scores 0. This shows that I enjoy Action and SciFi movies more than the other genres.

**13. Normalize the User Profile**

In [76]:
user_profile_normalized = (
user_profile / user_profile.sum()
)
user_profile_normalized

Action     0.500000
Comedy     0.000000
Drama      0.071429
Romance    0.071429
SciFi      0.357143
dtype: float64

**14. Score and Rank every movie in the dataset using the user profile**

In [77]:
movies["recommendation_score"] = (
movie_features
.mul(user_profile_normalized)
.sum(axis=1)
)
movies[["title", "recommendation_score"]]
ranked_movies = movies.sort_values(
"recommendation_score",
ascending=False
)
ranked_movies[["title", "recommendation_score"]]

,title,recommendation_score
0,Galaxy Mission,0.857143
4,Future World,0.857143
7,Space Warriors,0.857143
6,City Detectives,0.571429
2,Final Battle,0.500000
1,Love in Nairobi,0.142857
5,Broken Hearts,0.142857
8,Wedding Weekend,0.071429
9,Hidden Truth,0.071429
3,Family Weekend,0.000000


**15. Remove movies that the user has already watched**

In [78]:
watched_movies = ratings["movie_id"]
recommendations = ranked_movies[
~ranked_movies["movie_id"].isin(watched_movies)
]
recommendations[["title", "recommendation_score"]]

,title,recommendation_score
7,Space Warriors,0.857143
6,City Detectives,0.571429
5,Broken Hearts,0.142857
8,Wedding Weekend,0.071429
9,Hidden Truth,0.071429
3,Family Weekend,0.000000


**16. Generate Top 3 Recommendations**

In [79]:
top_3 = recommendations[
["title", "recommendation_score"]
].head(3)
top_3

,title,recommendation_score
7,Space Warriors,0.857143
6,City Detectives,0.571429
5,Broken Hearts,0.142857


**17. Explain the Recommendations**

1. Space Warriors: The model recommended this movie because it has the highest recommendation score 0.86 and matches my strong preference for Action and SciFi movies.

2. City Detectives: This movie has a recommendation score of 0.57, suggesting that it matches some of my preferred features, especially Action.

3. Broken Hearts: The model recommended this movie with a lower score of 0.14, meaning it has less similarity to my strongest preferences for Action and SciFi.

**18. Biuld a reusable function to generate recommendations for any user**

In [82]:
def recommend_movies(movies, ratings, features, n=3):
    user_movies = ratings.merge(movies, on="movie_id")
    rated_features = user_movies[features]
    weighted_features = rated_features.mul(user_movies["rating"], axis=0)
    user_profile = weighted_features.sum()
    user_profile = user_profile / user_profile.sum()
    scores = movies[features].mul(user_profile).sum(axis=1)
    results = movies.copy()
    results["recommendation_score"] = scores
    results = results[~results["movie_id"].isin(ratings["movie_id"])]
    return results.sort_values("recommendation_score", ascending=False).head(n)

recommend_movies(movies.drop(columns="recommendation_score"), ratings, features)

,movie_id,title,Action,Comedy,Drama,Romance,SciFi,recommendation_score
7,8,Space Warriors,1,0,0,0,1,0.857143
6,7,City Detectives,1,0,1,0,0,0.571429
5,6,Broken Hearts,0,0,1,1,0,0.142857


**19. Test Another User B**

In [83]:
user_b_ratings = pd.DataFrame({
"movie_id": [2, 4, 6, 9],
"rating": [5, 4, 5, 4]
})
recommend_movies(
movies,
user_b_ratings,
features
)

,movie_id,title,Action,Comedy,Drama,Romance,SciFi,recommendation_score
9,10,Hidden Truth,0,0,1,0,0,0.3125
6,7,City Detectives,1,0,1,0,0,0.3125
2,3,Final Battle,1,0,0,0,0,0.0000


## Compare User A and User B

No, User A and User B received different recommendations because their movie preferences are different. User A received Action and SciFi movies, such as Space Warriors and Robot Uprising, while User B received movies related to Romance, Drama, and Comedy. The model uses each user's ratings and movie genres to build a preference profile, which leads to different recommendations.

## Thinking Exercise: User C

1. What does the model learn about User C?

The model learns that User C prefers Action and SciFi because Galaxy Mission belongs to these two genres and received a rating of 5 stars.

2. Which features dominate User C's profile?

Action and SciFi have the highest scores, each with a normalized value of 0.50. The other genres score 0 because the only rated movie does not belong to them.

3. What could go wrong with the recommendations?

The recommendations may become repetitive because the model focuses only on Action and SciFi. Several movies may also receive similar scores, and the model cannot determine whether User C enjoys other genres.

4. Is one rating enough to understand User C's preferences?

No, one rating is not enough to understand User C's overall preferences. The model has limited information, which creates a cold-start problem and makes it difficult to provide diverse and accurate recommendations.

**20. Add Your Own Movie**

In [84]:
new_movie = {
"movie_id": 11,
"title": "Your Unique Movie",
"Action": 0,
"Comedy": 1,
"Drama": 1,
"Romance": 0,
"SciFi": 0
}
movies = pd.concat(
[movies, pd.DataFrame([new_movie])],
ignore_index=True
)

**21. Create Your Own User**

In [85]:
my_user = pd.DataFrame({
"movie_id": [1, 3, 6, 9],
"rating": [5, 4, 2, 3]
})
recommend_movies(
movies,
my_user,
features,
n=5
)

,movie_id,title,Action,Comedy,Drama,Romance,SciFi,recommendation_score
4,5,Future World,1,0,0,0,1,0.583333
7,8,Space Warriors,1,0,0,0,1,0.583333
6,7,City Detectives,1,0,1,0,0,0.458333
1,2,Love in Nairobi,0,0,1,1,0,0.291667
10,11,Your Unique Movie,0,1,1,0,0,0.208333


## 22. Interpretation

1. **What are my user's two strongest preferences?**

   Action 9 is the strongest preference, followed by Romance and SciFi, which are tied at 5 each.

2. **What are the three highest-scoring movies?**

   Island Escape ranks first with 0.708, followed by Star Voyagers and Future World at 0.583. Robot Uprising and Space Warriors also share the same score of 0.583.

3. **Why did the top movie rank first?**

   Island Escape combines Action, Comedy, and Romance. It matches several of the user's preferred genres, giving it a total score of 0.708, which is higher than the Action and SciFi movies that score 0.583.

4. **Was the result surprising? Why?**

   Yes, I expected an Action and SciFi movie to rank first because the user rated those genres highly. However, Island Escape ranked first because it combines more genres that contribute to the user's profile.

5. **What does the model ignore?**

   The model does not consider actors, directors, release years, languages, or movie descriptions. It also treats a low rating, such as 2 for Broken Hearts, as a positive contribution to the Drama and Romance scores instead of interpreting it as a possible dislike.

## 23. Limitations

1. **What are the limitations of using only five genre features?**

   The model considers only five genres, meaning movies with the same genre combinations can receive identical recommendation scores, even if their stories and other characteristics differ.

2. **What is the cold-start problem?**

   The cold-start problem occurs when a new user has few or no ratings. The model then has limited information to understand their preferences and provide accurate recommendations.

3. **What is the risk of narrow or repetitive recommendations?**

   The model mainly recommends movies that match the user's existing preferences. As a result, it may repeatedly suggest similar genres instead of introducing the user to new ones.

4. **How can poor movie metadata affect the recommendations?**

   Missing or incorrect genre information can produce inaccurate user profiles and recommendation scores, leading to unsuitable suggestions.

## 24. Conclusion

**Should the recommender give the same recommendations to different users? Explain why or why not.**

No, the recommender should not necessarily give different users the same recommendations because their ratings influence their individual preference profiles. For example, one user may rate Galaxy Mission 5 stars and Love in Nairobi 2 stars, making Action and SciFi more influential. Another user who gives the opposite ratings will have a profile that favours Drama and Romance. Therefore, their recommendation rankings may differ, although some movies could still appear in both lists. The model excludes movies that each user has already rated from their respective recommendations.